# 📥 Notebook 01 — Data Ingestion

**Project:** Nigeria Disease Surveillance Dashboard  
**Purpose:** Extract raw data from all sources and save to `data/raw/`.  
No cleaning happens here — this notebook is extraction only.  

**Sources covered:**
- NCDC Nigeria PDF Situation Reports
- WHO AFRO CSV/Excel files
- NASA POWER Rainfall API
- HDX Nigeria Health Facilities CSV
- NBS / WorldPop Population data
- GRID3 Nigeria State Shapefiles

**Output:** Raw files saved to `data/raw/`

---
> **Rule:** Never modify files in `data/raw/`. They are the ground truth.  
> All cleaning happens in Notebook 02.

## 1. Environment Setup

In [ ]:
import os
import sys
import time
import json
import logging
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

# Add the project root to sys.path so we can import src modules
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import project utilities
from src.utils.config import Paths, settings, Diseases
from src.utils.logger import get_logger, set_log_level
from src.utils.state_maps import CANONICAL_STATES, STATE_CENTROIDS

# Set log level for notebook output
set_log_level('INFO')
logger = get_logger('notebook_01')

# Ensure all data directories exist
Paths.ensure_all()

print('✅ Environment ready')
print(f'   Project root : {PROJECT_ROOT}')
print(f'   Raw data     : {Paths.raw}')
print(f'   Shapefiles   : {Paths.shapefiles}')
print(f'   Diseases     : {Diseases.all}')

✅ Environment ready
   Project root : /path/to/nigeria-disease-surveillance
   Raw data     : /path/to/data/raw
   Shapefiles   : /path/to/data/shapefiles
   Diseases     : ['Cholera', 'Lassa Fever', 'Mpox', 'Meningitis', 'Yellow Fever']


## 2. NCDC PDF Extraction

The NCDC publishes weekly Situation Reports as PDFs.  
Each PDF contains state-level case count tables for one or more diseases.

**Before running this cell:**
1. Download PDFs from [ncdc.gov.ng/diseases/sitreps](https://ncdc.gov.ng/diseases/sitreps)
2. Place them in the correct folder:
   - Cholera → `data/raw/ncdc_pdfs/cholera/`
   - Lassa Fever → `data/raw/ncdc_pdfs/lassa_fever/`
   - Mpox → `data/raw/ncdc_pdfs/mpox/`
   - Meningitis → `data/raw/ncdc_pdfs/meningitis/`
   - Yellow Fever → `data/raw/ncdc_pdfs/yellow_fever/`

> **Expect messy output.** NCDC PDFs have inconsistent formatting  
> across years. That messiness is handled in Notebook 02.

In [ ]:
from src.etl.extract import extract_ncdc_pdfs

raw_ncdc = {}

for disease_name, folder_key in Diseases.pdf_folder_map.items():
    folder = Paths.raw / 'ncdc_pdfs' / folder_key
    print(f'\n📂 Extracting {disease_name}...')
    print(f'   Looking in: {folder}')

    df = extract_ncdc_pdfs(folder, disease_name)

    if df.empty:
        print(f'   ⚠️  No PDFs found — place NCDC sitreps in {folder}')
    else:
        raw_ncdc[disease_name] = df
        print(f'   ✅ {len(df):,} rows extracted')
        print(f'   Columns: {list(df.columns)}')

print(f'\n📊 Extraction summary: {len(raw_ncdc)}/{len(Diseases.all)} diseases extracted')

### 2.1 Inspect raw NCDC output

Take a close look at what came out of the PDFs. Expect None values, garbled columns, and merged cells.

In [ ]:
def inspect_df(df: pd.DataFrame, name: str, n_rows: int = 3) -> None:
    """Print a structured summary of any DataFrame."""
    print(f'\n{'='*55}')
    print(f'  {name}')
    print(f'{'='*55}')
    print(f'  Shape   : {df.shape[0]:,} rows × {df.shape[1]} columns')
    print(f'  Columns : {list(df.columns)}')
    nulls = df.isnull().sum()
    if nulls.any():
        print(f'  Nulls   :\n{nulls[nulls > 0].to_string(indent=10)}')
    print(f'  Sample ({n_rows} rows):')
    display(df.head(n_rows))

for disease, df in raw_ncdc.items():
    inspect_df(df, f'NCDC — {disease}')

## 3. WHO AFRO Data

WHO AFRO data serves as a cross-validation source for NCDC figures.  

**Download from:** [afro.who.int](https://afro.who.int/health-topics)  
**Save to:** `data/raw/who/`  
**Supported formats:** `.csv`, `.xlsx`

In [ ]:
from src.etl.extract import extract_who_data

print('📂 Loading WHO AFRO data...')
raw_who = extract_who_data()

if raw_who.empty:
    print('⚠️  No WHO data found.')
    print('   Download from afro.who.int and place in data/raw/who/')
else:
    inspect_df(raw_who, 'WHO AFRO')

## 4. NASA POWER Rainfall API

We query the NASA POWER API for monthly precipitation at each state's centroid.  
This data is used to correlate rainfall with cholera and meningitis seasonality.

**No API key required.** Rate limit: ~30 requests/minute.  
The fetch loop adds a 2-second delay between states.

> ⏱️ **Expected runtime: 2–3 minutes** for all 37 states.

In [ ]:
from src.etl.extract import extract_nasa_rainfall

print('🌧️  Fetching NASA POWER rainfall data...')
print(f'   States to fetch: {len(STATE_CENTROIDS)}')
print('   This will take ~3 minutes.\n')

raw_rainfall = extract_nasa_rainfall(
    start_year     = 2015,
    end_year       = 2024,
    force_download = False,   # Use cached file if available
)

if raw_rainfall.empty:
    print('⚠️  No rainfall data retrieved (check network access)')
else:
    inspect_df(raw_rainfall, 'NASA POWER Rainfall')
    print(f'\n   Date range: {raw_rainfall["year"].min()}–{raw_rainfall["year"].max()}')
    print(f'   States    : {raw_rainfall["state"].nunique()}/37')
    print(f'   Avg mm/mo : {raw_rainfall["rainfall_mm"].mean():.1f}')

## 5. Health Facilities (HDX)

Facility locations are used for the accessibility gap analysis —  
identifying states with high disease burden but few facilities.

**Download from:** [data.humdata.org](https://data.humdata.org)  
**Search:** 'Nigeria health facilities'  
**Save to:** `data/raw/health_facilities.csv`

In [ ]:
from src.etl.extract import extract_health_facilities

print('🏥 Loading health facilities...')
raw_facilities = extract_health_facilities()

if raw_facilities.empty:
    print('⚠️  health_facilities.csv not found.')
    print('   Download from data.humdata.org (search Nigeria health facilities)')
else:
    inspect_df(raw_facilities, 'Health Facilities')
    if 'facility_type' in raw_facilities.columns:
        print('\n   Facility types:')
        print(raw_facilities['facility_type'].value_counts().head(10).to_string(indent=4))

## 6. Population Data (NBS / WorldPop)

State population estimates are used to calculate incidence rates per 100,000.

**Download from:** [nigerianstat.gov.ng](https://nigerianstat.gov.ng) or  
[worldpop.org](https://worldpop.org)  
**Save to:** `data/raw/nigeria_population.xlsx` or `.csv`

In [ ]:
from src.etl.extract import extract_population

print('👥 Loading population data...')
raw_population = extract_population()

if raw_population.empty:
    print('⚠️  Population file not found.')
    print('   Download from nigerianstat.gov.ng')
else:
    inspect_df(raw_population, 'Population Data')

## 7. Geospatial Shapefiles (GRID3)

State boundary polygons are used for choropleth maps.

**Download from:** [grid3.org](https://grid3.org) or [gadm.org/country/NGA](https://gadm.org/country/NGA)  
**Save to:** `data/shapefiles/nigeria_states.shp` (and associated files)

In [ ]:
from src.etl.extract import extract_shapefiles
import matplotlib.pyplot as plt

print('🗺️  Loading shapefiles...')
shapefiles = extract_shapefiles()

if not shapefiles:
    print('⚠️  No shapefiles found in data/shapefiles/')
    print('   Download from grid3.org')
else:
    for name, gdf in shapefiles.items():
        print(f'\n  ✅ {name}: {len(gdf)} features, CRS={gdf.crs}')
        print(f'     Columns: {list(gdf.columns)}')

    # Plot the state boundaries — your first map!
    if 'states' in shapefiles:
        fig, ax = plt.subplots(1, 1, figsize=(12, 8))
        shapefiles['states'].plot(
            ax          = ax,
            color       = '#E1F5EE',
            edgecolor   = '#1D9E75',
            linewidth   = 0.8,
        )
        ax.set_title(
            'Nigeria — State Boundaries (GRID3)',
            fontsize=14, fontweight='bold', pad=15
        )
        ax.set_xlabel('Longitude')
        ax.set_ylabel('Latitude')
        ax.axis('off')
        plt.tight_layout()
        # Save as PNG for the README
        fig.savefig(Paths.raw / 'nigeria_states_map.png',
                    dpi=150, bbox_inches='tight')
        plt.show()
        print('\n   ✅ Map saved to data/raw/nigeria_states_map.png')

## 8. Ingestion Summary

In [ ]:
print('='*55)
print('  NOTEBOOK 01 — INGESTION SUMMARY')
print('='*55)
print(f'  Timestamp: {datetime.now().strftime("%Y-%m-%d %H:%M")}')
print()

# Summarise what was loaded
sources = {
    'NCDC diseases extracted' : f'{len(raw_ncdc)}/{len(Diseases.all)}',
    'WHO AFRO rows'           : f'{len(raw_who):,}' if not raw_who.empty else 'NOT LOADED',
    'Rainfall records'        : f'{len(raw_rainfall):,}' if not raw_rainfall.empty else 'NOT LOADED',
    'Health facilities'       : f'{len(raw_facilities):,}' if not raw_facilities.empty else 'NOT LOADED',
    'Population rows'         : f'{len(raw_population):,}' if not raw_population.empty else 'NOT LOADED',
    'Shapefiles loaded'       : f'{len(shapefiles)} ({list(shapefiles.keys())})',
}

for label, value in sources.items():
    status = '✅' if 'NOT' not in str(value) else '⚠️ '
    print(f'  {status}  {label:<30} {value}')

# Count raw files saved
saved = list(Paths.raw.glob('*.csv'))
print(f'\n  Raw CSV files saved to data/raw/ : {len(saved)}')
for f in saved:
    print(f'    - {f.name} ({f.stat().st_size / 1024:.1f} KB)')

print()
print('  ➡️  Next step: Run Notebook 02 — Data Cleaning')
print('='*55)